# Reproducible analysis — *Pharmaceutical Research* (Revision 2)
**Employing General-Purpose and Biomedical LLMs with Advanced Prompt Engineering for Pharmacoepidemiologic Study Design**

End-to-end script requested by Reviewer #2 (Comment 5): reproduces **every reported statistic, effect size and table** from the three released workbooks, and adds the analyses requested during revision:

* **Data-flow table** and differential-missingness accounting (Comment 4)
* **Audited data corrections** (column-alignment in *sentinel-gpt-ltm* Q7; spurious spillover / 16th rows)
* **Clustered ordinal analysis** (protocol-level clustering) + **paired** prompt test + **cluster bootstrap** (Comment 5)
* **Depth-of-reasoning** three-category distribution + **sensitivity** for *partly agree* (Comment 3)
* **Set-based ontology metrics** (precision / recall / F1 / Jaccard / exact-set) by configuration and coding system, with native-vs-OMOP *convention* cases reported separately (Comment 8)

**To run in Colab:** upload `recreated_excel_file.xlsx`, `Logic_assessment.xlsx`, `Ontology__1_.xlsx`, then *Runtime → Run all*.

In [ ]:
!pip -q install statsmodels >/dev/null
import pandas as pd, numpy as np, re, warnings, os
warnings.filterwarnings('ignore'); from scipy import stats; import statsmodels.api as sm

# --- released filenames (upload these in Colab) ---
REL='/content/recreated_excel_file_corrected.xlsx'   # relevance (Q*.Likert.scale) + GPT-4o depth-of-reasoning
LOG='/content/Logic_assessment_corrected.xlsx'       # human logic (Q* human Likert scale)
ONT='/content/Ontology__1.xlsx'           # ontology-code mapping
PROT={'DARWIN':16,'HMA-EMA':15,'Sentinel':15}; QN=9   # 46 protocols x 9 questions

In [ ]:
def parse_sheet(sh):
    U=sh.upper()
    src='HMA-EMA' if ('HMA' in U or 'EMA' in U) else 'DARWIN' if 'DARWIN' in U else 'Sentinel' if 'SENTINEL' in U else None
    llm=('GPT-4o' if 'GPT' in U else 'DeepSeek-R1' if 'DEEPSEEK' in U else
         'Qwen2-med' if 'IRATH' in U else 'BioLlama' if 'LLAMA' in U else None)
    prm='ACT' if 'ACT' in U else 'LTM' if 'LTM' in U else None
    cls='Biomedical' if llm in ('Qwen2-med','BioLlama') else 'Non-Biomedical'
    return src,llm,prm,cls
def find(cols,n,*keys):
    for c in cols:
        s=str(c).upper()
        if re.search(rf"\bQ0*{n}\b",s) and all(k.upper() in s for k in keys): return c
def codes(x):
    if pd.isna(x): return set()
    return {c.strip() for c in re.split(r"[;,/]| +",str(x)) if c.strip() and c.strip().lower()!="nan"}

# DATA CORRECTIONS (audited):
# (1) restrict each sheet to its pre-specified protocol count -> removes spurious
#     spillover rows (DARWIN_GPT_LTM r17-22, blank Case.no) and the extra 16th rows
#     in sentinel-llama / hma-irath (logic workbook).
# (2) sentinel-gpt-ltm Q7 human-Likert column holds evaluator prose (one-column
#     displacement) -> coerced to numeric => NA (as it already was in all summaries).
def load_long(path, want):
    rows=[]; xl=pd.ExcelFile(path)
    for sh in xl.sheet_names:
        src,llm,prm,cls=parse_sheet(sh)
        if src is None: continue
        d=pd.read_excel(xl,sheet_name=sh).iloc[:PROT[src]].copy()
        for i,r in d.iterrows():
            for n in range(1,QN+1):
                rec=dict(source=src,llm=llm,prompt=prm,cls=cls,protocol=f"{src[:3]}{i+1}",q=n,cfg=f"{llm}-{prm}")
                if want=="rel":
                    lc=find(d.columns,n,"LIKERT"); dc=find(d.columns,n,"DEPTH")
                    rec["relevance"]=pd.to_numeric(r[lc],errors="coerce") if lc else np.nan
                    s=str(r[dc]).strip().lower() if dc and pd.notna(r[dc]) else ""
                    rec["dor"]=("Agree" if s.startswith("agree") else "Partly" if "part" in s
                                else "Disagree" if s.startswith("disagree") else None)
                else:
                    hc=find(d.columns,n,"HUMAN","LIKERT")
                    rec["logic"]=pd.to_numeric(r[hc],errors="coerce") if hc else np.nan
                rows.append(rec)
    return pd.DataFrame(rows)

rel=load_long(REL,"rel"); logic=load_long(LOG,"log")
print("relevance rows",len(rel),"| logic rows",len(logic))

relevance rows 2070 | logic rows 2070


In [ ]:
print(logic)

       source       llm prompt             cls protocol  q           cfg  \
0      DARWIN    GPT-4o    LTM  Non-Biomedical     DAR1  1    GPT-4o-LTM   
1      DARWIN    GPT-4o    LTM  Non-Biomedical     DAR1  2    GPT-4o-LTM   
2      DARWIN    GPT-4o    LTM  Non-Biomedical     DAR1  3    GPT-4o-LTM   
3      DARWIN    GPT-4o    LTM  Non-Biomedical     DAR1  4    GPT-4o-LTM   
4      DARWIN    GPT-4o    LTM  Non-Biomedical     DAR1  5    GPT-4o-LTM   
...       ...       ...    ...             ...      ... ..           ...   
2065  HMA-EMA  BioLlama    LTM      Biomedical    HMA15  5  BioLlama-LTM   
2066  HMA-EMA  BioLlama    LTM      Biomedical    HMA15  6  BioLlama-LTM   
2067  HMA-EMA  BioLlama    LTM      Biomedical    HMA15  7  BioLlama-LTM   
2068  HMA-EMA  BioLlama    LTM      Biomedical    HMA15  8  BioLlama-LTM   
2069  HMA-EMA  BioLlama    LTM      Biomedical    HMA15  9  BioLlama-LTM   

      logic  
0       NaN  
1       NaN  
2       NaN  
3       NaN  
4       NaN  
...

## 1. Data-flow table & differential missingness (Comment 4)

In [ ]:
df=[]
for (cfg,src),g in rel.groupby(["cfg","source"]):
    exp=PROT[src]*QN; lg=logic[(logic.cfg==cfg)&(logic.source==src)]
    df.append((cfg,src,exp,int(g.relevance.notna().sum()),int(g.dor.notna().sum()),int(lg.logic.notna().sum())))
DF=pd.DataFrame(df,columns=["config","source","expected","relevance","depthReason","humanLogic"])
DF["rel_miss"]=DF.expected-DF.relevance; DF["logic_miss"]=DF.expected-DF.humanLogic
display(DF)
for cls in ["Non-Biomedical","Biomedical"]:
    cf=rel[rel.cls==cls].cfg.unique(); s=DF[DF.config.isin(cf)]
    print(f"{cls}: relevance {s.relevance.sum()}/{s.expected.sum()} ({100*s.relevance.sum()/s.expected.sum():.1f}%),",
          f"human-logic {s.humanLogic.sum()}/{s.expected.sum()} ({100*s.humanLogic.sum()/s.expected.sum():.1f}%)")

,config,source,expected,relevance,depthReason,humanLogic,rel_miss,logic_miss
0,BioLlama-LTM,DARWIN,144,130,130,0,14,144
1,BioLlama-LTM,HMA-EMA,135,101,101,49,34,86
2,BioLlama-LTM,Sentinel,135,120,120,44,15,91
3,DeepSeek-R1-LTM,DARWIN,144,130,130,56,14,88
4,DeepSeek-R1-LTM,HMA-EMA,135,101,101,99,34,36
5,DeepSeek-R1-LTM,Sentinel,135,121,121,118,14,17
6,GPT-4o-ACT,DARWIN,144,130,130,129,14,15
7,GPT-4o-ACT,HMA-EMA,135,101,101,100,34,35
8,GPT-4o-ACT,Sentinel,135,121,121,117,14,18
9,GPT-4o-LTM,DARWIN,144,140,143,132,4,12


Non-Biomedical: relevance 1058/1242 (85.2%), human-logic 838/1242 (67.5%)
Biomedical: relevance 702/828 (84.8%), human-logic 122/828 (14.7%)


## 2. Relevance — clustered ordinal analysis (Comment 5)
Ordinal (cumulative-logit) GEE with **protocol-level clustering**; fixed effects for configuration, source, question. Class-level contrasts are reported **descriptively** (only two models per class).

In [ ]:
R=rel.dropna(subset=["relevance"]).copy(); R["relevance"]=R.relevance.astype(int); R["pid"]=R.source+"_"+R.protocol; R["cfg_c"]=R.cfg
print("Kruskal-Wallis across 5 configs (descriptive): H=%.1f"%stats.kruskal(*[g.relevance for _,g in R.groupby("cfg")])[0])
m=sm.OrdinalGEE.from_formula('relevance ~ C(cfg_c,Treatment(reference="GPT-4o-LTM"))+C(source)+q',
                             groups="pid",data=R,cov_struct=sm.cov_struct.Independence()).fit(maxiter=80)
for k in m.params.index:
    if "cfg_c" in k or "source" in k or k=="q":
        lab=k.split("T.")[-1].rstrip("]") if "T." in k else k
        print(f"  {lab:20s} beta={m.params[k]:+.2f}  p={m.pvalues[k]:.3f}")

Kruskal-Wallis across 5 configs (descriptive): H=384.3
  BioLlama-LTM         beta=-1.36  p=0.000
  DeepSeek-R1-LTM      beta=+0.01  p=0.933
  GPT-4o-ACT           beta=-0.06  p=0.694
  Qwen2-med-LTM        beta=-2.50  p=0.000
  HMA-EMA              beta=+1.40  p=0.000
  Sentinel             beta=-0.26  p=0.069
  q                    beta=-0.06  p=0.036


## 3. Prompt strategy LTM vs ACT — paired (Comments 5 & 7)
The GPT-4o-LTM vs GPT-4o-Active comparison is **paired** by protocol×question.

In [ ]:
g=R[R.llm=="GPT-4o"]; piv=g.pivot_table(index=["pid","q"],columns="prompt",values="relevance").dropna()
print("unpaired Mann-Whitney p=%.3f (originally reported)"%stats.mannwhitneyu(g[g.prompt=="LTM"].relevance,g[g.prompt=="ACT"].relevance)[1])
print("PAIRED Wilcoxon n=%d p=%.3f  -> no prompt effect"%(len(piv),stats.wilcoxon(piv.LTM,piv.ACT)[1]))
rng=np.random.default_rng(0); u=piv.reset_index().pid.unique(); byp={p:s for p,s in piv.reset_index().groupby("pid")}
bo=[pd.concat([byp[p] for p in rng.choice(u,len(u),True)]).eval("LTM-ACT").mean() for _ in range(2000)]
print("cluster-bootstrap mean diff 95%% CI=[%+.3f,%+.3f] (includes 0)"%(np.percentile(bo,2.5),np.percentile(bo,97.5)))

unpaired Mann-Whitney p=0.549 (originally reported)
PAIRED Wilcoxon n=342 p=0.927  -> no prompt effect
cluster-bootstrap mean diff 95% CI=[-0.164,+0.155] (includes 0)


## 4. Depth-of-reasoning (GPT-4o self-assessed) — 3 categories + sensitivity (Comment 3)
**Distinct construct** from the human logic ratings in Figures 4–6. The originally reported 70.7% excluded *partly agree* from the denominator; with the denominator rule disclosed and sensitivity analyses:

In [ ]:
# reasoning-concordance is its own variable: count ALL labelled cells
# (independent of relevance completeness) -> uses rel, not R
tab=rel.dropna(subset=["dor"]).groupby(["cls","dor"]).size().unstack(fill_value=0).reindex(columns=["Agree","Disagree","Partly"]); display(tab)
for name,acc in [("exclude-partly (as reported)",None),("partly=acceptable",True),("partly=unacceptable",False)]:
    res=[]
    for cls in ["Non-Biomedical","Biomedical"]:
        a,d,p_=tab.loc[cls]
        num,den=(a,a+d) if acc is None else ((a+p_,a+d+p_) if acc else (a,a+d+p_)); res.append((num,den))
    chi=stats.chi2_contingency([[res[0][0],res[0][1]-res[0][0]],[res[1][0],res[1][1]-res[1][0]]])[0]
    print(f"{name:28s}: GP={100*res[0][0]/res[0][1]:.1f}% ({res[0][0]}/{res[0][1]}), Bio={100*res[1][0]/res[1][1]:.1f}% ({res[1][0]}/{res[1][1]}), chi2={chi:.0f}")

dor,Agree,Disagree,Partly
cls,,,
Biomedical,135,533,34
Non-Biomedical,705,292,110


exclude-partly (as reported): GP=70.7% (705/997), Bio=20.2% (135/668), chi2=406
partly=acceptable           : GP=73.6% (815/1107), Bio=24.1% (169/702), chi2=423
partly=unacceptable         : GP=63.7% (705/1107), Bio=19.2% (135/702), chi2=340


## 5. Human logic — differential missingness (Comment 4)
Coverage differs hugely by class (Non-Biomedical ≈68% vs Biomedical ≈15%); Figures 4–6 must be read with caution and Figure 5's DARWIN×biomedical cell has no data.

In [ ]:
L=logic.dropna(subset=["logic"])
for cls in ["Non-Biomedical","Biomedical"]:
    v=L[L.cls==cls]; print(f"{cls}: n={len(v)}, median={v.logic.median()}, mean={v.logic.mean():.2f}")

Non-Biomedical: n=838, median=5.0, mean=4.56
Biomedical: n=122, median=5.0, mean=4.16


## 6. Ontology-code mapping — set-based metrics (Comment 8)
Precision / recall / F1 / Jaccard / exact-set by configuration and coding system. Native SNOMED CT / RxNorm identifiers evaluated against OMOP-concept-ID references are **coding-convention** differences and are reported **separately** (they require a terminology crosswalk to score). The taxonomy covers **3 of 5** configurations because the ontology workbook contains only the GPT-4o and DeepSeek columns.

In [ ]:
def fam(o):
    o=str(o).upper()
    for k in ["ATC","RXNORM","SNOMED","HCPCS","CPT","ICD","READ","SMG"]:
        if k in o: return "RxNorm" if k=="RXNORM" else k
    return "other"
CFG={"GPT-4 LTM":"GPT-4o-LTM","GPT-4 ACT":"GPT-4o-ACT","Deepseek LTM":"DeepSeek-R1-LTM"}
rows=[]; xlo=pd.ExcelFile(ONT)
for sh in xlo.sheet_names:
    d=pd.read_excel(xlo,sheet_name=sh)
    for _,r in d.iterrows():
        if pd.isna(r.get("code")): continue
        ref=codes(r["code"]); f=fam(r["ontology"])
        for col,cfg in CFG.items():
            pred=codes(r.get(col)); inter=ref&pred; uni=ref|pred
            prec=len(inter)/len(pred) if pred else np.nan
            rec=len(inter)/len(ref) if ref else np.nan
            f1=2*prec*rec/(prec+rec) if pred and ref and (prec+rec)>0 else (0.0 if pred else np.nan)
            conv=int(f in("SNOMED","RxNorm") and len(pred)>0 and len(inter)==0 and all(c.isdigit() for c in pred))
            rows.append((cfg,f,prec,rec,f1,len(inter)/len(uni) if uni else np.nan,int(ref==pred and len(pred)>0),conv,int(not pred)))
M=pd.DataFrame(rows,columns=["cfg","system","precision","recall","F1","jaccard","exact","convention","omitted"]); att=M[M.omitted==0]
print("by configuration (attempted):"); display(att.groupby("cfg")[["precision","recall","F1","jaccard","exact"]].mean().round(3))
print("by coding system (attempted, pooled):")
display(att.groupby("system")[["precision","recall","F1","jaccard"]].mean().round(3).assign(convention=M.groupby("system").convention.sum()).fillna(0))
print(f"native-vs-OMOP convention cases (reported separately): {int(M.convention.sum())}")
print(f"omissions: {int(M.omitted.sum())}/{len(M)} ({100*M.omitted.mean():.1f}%)")

by configuration (attempted):


,precision,recall,F1,jaccard,exact
cfg,,,,,
DeepSeek-R1-LTM,0.393,0.248,0.279,0.233,0.123
GPT-4o-ACT,0.467,0.283,0.322,0.272,0.139
GPT-4o-LTM,0.515,0.282,0.326,0.270,0.125


by coding system (attempted, pooled):


,precision,recall,F1,jaccard,convention
system,,,,,
ATC,0.647,0.275,0.346,0.252,0
CPT,0.783,0.280,0.370,0.267,0
HCPCS,0.607,0.236,0.322,0.203,0
ICD,0.661,0.455,0.499,0.439,0
READ,0.000,0.000,0.000,0.000,0
RxNorm,0.000,0.000,0.000,0.000,28
SNOMED,0.000,0.000,0.000,0.000,32


native-vs-OMOP convention cases (reported separately): 60
omissions: 268/477 (56.2%)
